# Contract RAG Dev Notebook
Iterative development of hybrid retrieval, agentic RAG, guardrails, and RAGAS evaluation  
on top of the existing FAISS pipeline.

**Run cells top-to-bottom on first pass. After that, any section is independently runnable.**

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, logging
sys.path.insert(0, os.path.abspath('.'))

logging.basicConfig(
    level=logging.INFO,
    format='%(name)s | %(levelname)s | %(message)s',
)

from dotenv import load_dotenv
load_dotenv()
print('Environment loaded')

---
## Step 0 — Verify Existing Pipeline

**Pre-confirmed findings:**

| Aspect | Value |
|--------|-------|
| FAISS index location | `faiss_index/` (single unified index) |
| Embedding model | `text-embedding-3-small` (LangChain OpenAIEmbeddings) |
| Chunk metadata | `doc_id` ✓  `filename` ✓  `page` (1-indexed) ✓  `chunk_index` ✓ |
| Per-document index? | No — single index, `doc_id` filtered at query time |
| `[doc_id, page]` citations | Supported — both fields on every chunk |

Run the cell below to confirm live index state before proceeding.

In [ ]:
from docs.pipeline import get_vector_store

vs = get_vector_store()
all_docs = list(vs.vector_store.docstore._dict.values())

print(f'FAISS index loaded  --  {len(all_docs)} chunks')

if all_docs:
    sample = all_docs[0]
    print(f'\nSample metadata: {sample.metadata}')
    print(f'\nChunk preview:   {repr(sample.page_content[:200])}')
else:
    print('\nWARNING: Index is empty -- upload at least one PDF via POST /upload/stream before running retrieval cells.')

---
## Section 1 — Knowledge RAG (Hybrid + Cross-Encoder Reranking)

**Pipeline:**
```
Query
 -> EnsembleRetriever (BM25 0.4 + dense FAISS 0.6)   [top K_INITIAL]
      -> CrossEncoder reranker (ms-marco-MiniLM-L-6-v2)  [top K_FINAL]
           -> LLM (gpt-4o, T=0)  [doc_id, page] citations enforced
                -> { answer: str, sources: [{doc_id, page, filename}] }
```

**Why hybrid?**  
Pure embedding similarity misses exact contract terms like *"Net-30"*, *"Force Majeure"*, *"Effective Date"*.  
BM25 catches those; dense handles semantics. Cross-encoder then re-scores all candidates jointly.

In [ ]:
# Tuning knobs -- adjust here and rerun any cell below
K_INITIAL  = 20    # candidates pulled from hybrid retriever
K_FINAL    = 5     # kept after cross-encoder reranking
BM25_W     = 0.4   # BM25 weight in EnsembleRetriever (RRF)
DENSE_W    = 0.6   # dense (FAISS) weight
LLM_MODEL  = 'gpt-4o'

### 1a. Build Hybrid Retriever

In [ ]:
from src.agents.retriever import build_hybrid_retriever, rerank

# Rebuild on each call so BM25 reflects the current index state
hybrid_retriever = build_hybrid_retriever(
    vs,
    k=K_INITIAL,
    bm25_weight=BM25_W,
    dense_weight=DENSE_W,
)
print('Hybrid retriever ready')

In [ ]:
# Sanity-check: raw retrieval then reranking
test_q = 'What is the payment term?'

raw_docs  = hybrid_retriever.invoke(test_q)
reranked  = rerank(test_q, raw_docs, top_n=K_FINAL)

print(f'Raw candidates : {len(raw_docs)}')
print(f'After reranking: {len(reranked)}')
print()

for i, doc in enumerate(reranked, 1):
    m       = doc.metadata
    preview = doc.page_content[:140].replace('\n', ' ')
    print(f'  {i}. [{m.get("filename")} p.{m.get("page")}]  doc_id={str(m.get("doc_id", ""))[:8]}...')
    print(f'     {preview}')
    print()

### 1b. Full Knowledge RAG — answer with `[doc_id, page]` inline citations

In [ ]:
from openai import OpenAI
from src.agents.knowledge_rag import EnhancedKnowledgeRAG

client = OpenAI()

krag = EnhancedKnowledgeRAG(
    client=client,
    vector_store=vs,
    model=LLM_MODEL,
    k_initial=K_INITIAL,
    k_final=K_FINAL,
    bm25_weight=BM25_W,
    dense_weight=DENSE_W,
)
print('EnhancedKnowledgeRAG ready')

In [ ]:
# Edit these to match clauses in your actual contracts
SAMPLE_QUESTIONS = [
    'What is the payment term?',
    'What is the governing law?',
    'Explain the force majeure clause.',
    'What is the notice period for termination?',
]

for q in SAMPLE_QUESTIONS:
    result = krag.run(q)
    print('=' * 70)
    print(f'Q: {q}')
    print(f'A: {result["answer"]}')
    print('Sources:')
    for s in result['sources']:
        short_id = str(s.get('doc_id') or '')[:8]
        print(f'  - {s["filename"]}  p.{s["page"]}  doc_id={short_id}...')
    print()

---
### Section 1 complete

Review the output above:
- Every answer should contain `[doc_id, page]` inline citations
- Sources list should match the cited pages
- If answers are wrong or missing, try raising `K_INITIAL` / lowering `K_FINAL`,  
  or adjusting `BM25_W` / `DENSE_W` in the knobs cell above

**Confirm before proceeding to Section 2 (Agentic RAG — LangGraph).**

---